<a href="https://colab.research.google.com/github/DemianH9/agente-sapu-isy0101/blob/main/ep3_observabilidad/agente_sapu_EP3_observabilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EP3 · Observabilidad del Agente Clínico SAPU
**ISY0101 · Ingeniería de Soluciones con IA** — continuación de la EP1 (RAG) y la EP2 (agente LangChain con 4 herramientas).

| Sección | Indicador | Contenido |
|---|---|---|
| 3–4 | IE1, IE2 | Agente v1 (EP2 sin cambios) instrumentado con `TrazaHandler` y `ejecutar_observado()` |
| 5–6 | IE1 | 20 casos con respuesta esperada en 9 escenarios de variabilidad; pruebas deterministas |
| 7 | IE1, IE2 | Corrida v1 con T = 0,0 y T = 0,7 (3 repeticiones c/u = 120 ejecuciones) |
| 8–9 | IE5 | Agente v2 (R1–R3) y tabla `comparacion` |
| 10 | IE3 | Análisis de registros y trazas: fallas, tiempo por span, traza más lenta, WARNING/ERROR |
| 11–12 | IE4, IE5 | Figuras y exportación de logs para el dashboard Streamlit |

**Requisito:** Secret `LLM_API_KEY` (Mistral) con acceso habilitado para este notebook. Duración aproximada: 20–30 min.

## 1. Instalación

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    if not os.path.exists("/content/agente-sapu-isy0101"):
        !git clone -q https://github.com/DemianH9/agente-sapu-isy0101 /content/agente-sapu-isy0101
    %cd /content/agente-sapu-isy0101/ep3_observabilidad
    !pip install -q -r requirements.txt
print(os.getcwd())

## 2. Configuración

In [ ]:
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass
except ImportError:
    from dotenv import load_dotenv; load_dotenv()
os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
assert os.getenv("LLM_API_KEY"), "Falta el Secret LLM_API_KEY"
sys.path.insert(0, os.getcwd())

import pandas as pd
import agente, observabilidad as obs, evaluacion as ev
pd.set_option("display.max_colwidth", 120)
print("✅ Modelo:", os.getenv("LLM_MODEL"))

## 3. Agente v1 (EP2 sin cambios)
`agente.crear_herramientas(..., "v1")` reproduce las 4 herramientas, el guardrail (regex de RUT con guion) y el prompt del notebook `notebooks/agente_sapu_IL2_1.ipynb`.

In [ ]:
embeddings = agente.crear_embeddings()
vector_db = agente.crear_vector_db(embeddings)
tools_v1 = agente.crear_herramientas(vector_db, "v1")

def executor(version, temperatura):
    tools = tools_v1 if version == "v1" else tools_v2
    return agente.crear_agente(agente.crear_llm(temperatura), tools, version, max_iterations=6)

print("✅ Herramientas v1:", [t.name for t in tools_v1])

## 4. Observabilidad
`TrazaHandler` (callback de LangChain) crea un *span* por cada llamada al LLM y a cada herramienta. `ejecutar_observado()` aplica el guardrail, mide latencia, CPU, RAM, tokens y costo, reintenta ante *rate limit* y escribe `logs/ejecuciones.jsonl` y `logs/agente.log` después de enmascarar los datos personales.

In [ ]:
LOGS = pathlib.Path("logs"); LOGS.mkdir(exist_ok=True)
RUTA = LOGS / "ejecuciones.jsonl"
REINICIAR = False   # True borra los logs y repite todas las ejecuciones
if REINICIAR:
    for f in LOGS.glob("*"): f.unlink()
logger = obs.configurar_logging(LOGS)

# Prueba de humo: una consulta trazada (no se guarda en el set de resultados)
demo = obs.ejecutar_observado(executor("v1", 0.0), "¿Dosis de hidrocortisona para un niño de 15 kg?",
                              version="v1", guardrail=agente.contiene_datos_paciente_v1,
                              meta={"temperatura": 0.0, "caso_id": "demo"}, logger=logger,
                              ruta_jsonl=LOGS / "demo.jsonl")
print(demo["respuesta"], "\n")
pd.DataFrame(demo["spans"])[["tipo", "nombre", "inicio_s", "duracion_s", "estado", "tokens_entrada", "tokens_salida"]]

## 5. Casos de prueba (variabilidad de datos)

In [ ]:
casos = ev.cargar_casos("casos_prueba.json")
tabla_casos = pd.DataFrame([{"id": c["id"], "escenario": c["escenario"], "categoria": c["categoria"],
                             "consulta": c["consulta"]} for c in casos])
print(len(casos), "casos ·", tabla_casos["escenario"].nunique(), "escenarios")
tabla_casos

## 6. Pruebas deterministas (sin LLM)
Confirman H1 (guardrail v1) y H2 (vademécum v1 ante errores de tipeo) y validan la instrumentación.

In [ ]:
!python -m pytest -q tests -p no:cacheprovider

## 7. Corrida v1: T = 0,0 y T = 0,7 (3 repeticiones cada una)

In [ ]:
for T in (0.0, 0.7):
    ev.correr(executor("v1", T), casos, version="v1", temperatura=T, repeticiones=3,
              guardrail=agente.contiene_datos_paciente_v1, logger=logger, ruta_jsonl=RUTA)

In [ ]:
df = ev.cargar_registros(RUTA)
ev.tabla_metricas(df[df.version == "v1"]).round(3)

## 8. Agente v2 (recomendaciones R1–R3)
- **R1** guardrail ampliado (RUT en todo formato, correo, teléfono y nombres).
- **R2** vademécum tolerante a tipeo (similitud ≥ 0,8).
- **R3** prompt v2: pedir el peso, rechazar lo fuera de alcance y las órdenes de inventar.

In [ ]:
tools_v2 = agente.crear_herramientas(vector_db, "v2")
print(agente.SYSTEM_V2[len(agente.SYSTEM_V1):])

## 9. Corrida v2 (T = 0,0) y tabla `comparacion`

In [ ]:
ev.correr(executor("v2", 0.0), casos, version="v2", temperatura=0.0, repeticiones=3,
          guardrail=agente.contiene_pii_v2, logger=logger, ruta_jsonl=RUTA)

In [ ]:
df = ev.cargar_registros(RUTA)
RES = pathlib.Path("resultados"); RES.mkdir(exist_ok=True)
metr = ev.tabla_metricas(df); metr.to_csv(RES / "metricas.csv")
comparacion = ev.tabla_comparacion(df); comparacion.to_csv(RES / "comparacion.csv")
print(df.groupby("config").size().to_string(), "\n")
comparacion

In [ ]:
metr.round(3)

## 10. Análisis de registros y trazabilidad

In [ ]:
fallas = ev.fallas_por_chequeo(df)
print("Chequeos fallidos por configuración y caso:")
display(fallas.groupby(["config", "caso_id", "chequeo"]).size().unstack("config").fillna(0).astype(int))

spans = pd.DataFrame([dict(s, config=r.config, caso_id=r.caso_id, categoria=r.categoria)
                      for r in df.itertuples() for s in r.spans])
print("\nTiempo total por tipo de span (s):")
display(spans.groupby(["config", "tipo"])["duracion_s"].agg(["count", "sum", "mean"]).round(2))
print("\nEstado de las herramientas:")
display(spans[spans.tipo == "tool"].groupby(["config", "nombre", "estado"]).size().unstack("estado").fillna(0).astype(int))

lenta = df.sort_values("latencia_s", ascending=False).iloc[0]
print(f"\nTraza más lenta: {lenta.trace_id} · {lenta.config} · {lenta.caso_id} · {lenta.latencia_s:.2f} s")
display(pd.DataFrame(lenta.spans)[["tipo", "nombre", "inicio_s", "duracion_s", "estado"]])

niveles = pd.Series([l.split(" | ")[1].strip() for l in (LOGS / "agente.log").read_text(encoding="utf-8").splitlines() if " | " in l])
eventos = pd.Series([l.split(" | ")[2].split()[4].split("=")[0] for l in (LOGS / "agente.log").read_text(encoding="utf-8").splitlines()
                     if " | WARNING" in l or " | ERROR" in l])
print("\nNiveles en agente.log:", niveles.value_counts().to_dict())
print("Eventos WARNING/ERROR:", eventos.value_counts().to_dict())

In [ ]:
# Números para el informe (hallazgos H1–H5, recomendaciones R1–R3 y costo a escala)
g = lambda cfg: df[df.config == cfg]
v1_0, v1_7, v2_0 = g("v1 · T=0.0"), g("v1 · T=0.7"), g("v2 · T=0.0")
v1 = df[df.version == "v1"]
prec_esc = ev.precision_por_escenario(df)
sin_dato_c05 = sum(any(s["nombre"] == "consultar_vademecum" and s["estado"] == "sin_dato" for s in r.spans)
                   for r in v1[v1.caso_id == "C05"].itertuples())
calc_c06 = sum("calcular_dosis" in r.herramientas for r in v1[v1.caso_id == "C06"].itertuples())
dosis_v1 = v1_0[(v1_0.categoria == "dosis") & ~v1_0.bloqueado]
CONSULTAS_TURNO, TURNOS_MES = 50, 90   # supuesto: 50 consultas al agente por turno, 3 turnos/día x 30 días
m = {c: ev.metricas(g(c)) for c in ("v1 · T=0.0", "v1 · T=0.7", "v2 · T=0.0")}
hallazgos = {
    "H1_pii_enviada_v1": int(v1.pii_enviada_al_llm.sum()),
    "H1_casos_pii_enviada": sorted(v1[v1.pii_enviada_al_llm].caso_id.unique().tolist()),
    "H2_c05_sin_dato": f"{sin_dato_c05} de {len(v1[v1.caso_id == 'C05'])}",
    "H3_pct_tiempo_llm_v1_T0": round(m["v1 · T=0.0"]["llm_pct_tiempo"], 1),
    "H3_llamadas_llm_dosis_v1_T0": round(dosis_v1.llamadas_llm.mean(), 2),
    "H3_p95_dosis_v1_T0": round(dosis_v1.latencia_s.quantile(0.95), 2),
    "H4_c06_calcular_dosis": f"{calc_c06} de {len(v1[v1.caso_id == 'C06'])}",
    "H5_consistencia_T0_vs_T07": [round(m["v1 · T=0.0"]["consistencia_pct"], 1), round(m["v1 · T=0.7"]["consistencia_pct"], 1)],
    "H5_rate_limit_total": int(df.rate_limits.sum()),
    "H5_max_iteraciones_total": int((df.error == "max_iteraciones").sum()),
    "R1_guardrail_v1_v2": [round(m["v1 · T=0.0"]["guardrail_pct"], 1), round(m["v2 · T=0.0"]["guardrail_pct"], 1)],
    "R2_ruido_v1_v2": [round(prec_esc.loc["ruido", "v1 · T=0.0"], 1), round(prec_esc.loc["ruido", "v2 · T=0.0"], 1)],
    "R3_dato_faltante_v1_v2": [round(prec_esc.loc["dato_faltante", "v1 · T=0.0"], 1), round(prec_esc.loc["dato_faltante", "v2 · T=0.0"], 1)],
    "R3_prompt_injection_v1_v2": [round(prec_esc.loc["prompt_injection", "v1 · T=0.0"], 1), round(prec_esc.loc["prompt_injection", "v2 · T=0.0"], 1)],
    "costo_usd_1000_v1_T0": round(m["v1 · T=0.0"]["usd_1000"], 4),
    "costo_usd_1000_v2_T0": round(m["v2 · T=0.0"]["usd_1000"], 4),
    "costo_mes_usd_v2": round(m["v2 · T=0.0"]["usd_1000"] / 1000 * CONSULTAS_TURNO * TURNOS_MES, 2),
    "supuesto_costo": f"{CONSULTAS_TURNO} consultas por turno, {TURNOS_MES} turnos al mes",
    "traza_mas_lenta": {"trace_id": lenta.trace_id, "config": lenta.config, "caso": lenta.caso_id, "latencia_s": round(lenta.latencia_s, 2)},
}
(RES / "hallazgos.json").write_text(json.dumps(hallazgos, ensure_ascii=False, indent=2), encoding="utf-8")
prec_esc.round(1).to_csv(RES / "precision_por_escenario.csv")
print(json.dumps(hallazgos, ensure_ascii=False, indent=2))
prec_esc.round(1)

## 11. Figuras

In [ ]:
import matplotlib.pyplot as plt
FIG = pathlib.Path("figuras"); FIG.mkdir(exist_ok=True)
orden = ["base", "parafraseo", "ruido", "limite", "dato_faltante", "farmaco_ausente", "rut_formatos", "nombre_sin_rut", "prompt_injection"]
p = prec_esc.reindex(orden)[["v1 · T=0.0", "v2 · T=0.0"]]
ax = p.plot.bar(figsize=(9, 3.8), color=["#4C78A8", "#F58518"], width=0.75)
ax.set_ylabel("Precisión (%)"); ax.set_xlabel(""); ax.set_ylim(0, 105); ax.axhline(90, ls="--", c="gray", lw=1)
ax.set_title("Precisión por escenario de variabilidad, v1 vs v2 (T = 0)"); ax.legend(title=None, loc="lower left")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.savefig(FIG / "fig1_precision_escenario.png", dpi=200); plt.show()

al = df[~df.bloqueado]
fig, ax = plt.subplots(figsize=(9, 3.8))
datos = [al[al.config == c].latencia_s for c in ("v1 · T=0.0", "v1 · T=0.7", "v2 · T=0.0")]
ax.boxplot(datos, tick_labels=["v1 · T=0.0", "v1 · T=0.7", "v2 · T=0.0"]); ax.axhline(obs.SLO_LATENCIA_S, ls="--", c="red", lw=1)
ax.set_ylabel("Latencia (s)"); ax.set_title("Latencia por configuración (SLO 8 s)")
plt.tight_layout(); plt.savefig(FIG / "fig2_latencia.png", dpi=200); plt.show()

## 12. Exportar resultados
Copia `logs/`, `resultados/` y `figuras/` a Google Drive (`MiUnidad/EP3_SAPU/`) para abrir el dashboard (`streamlit run dashboard/app.py`). Los logs ya están enmascarados.

In [ ]:
import shutil
!zip -q -r ep3_resultados.zip logs resultados figuras -x "logs/demo.jsonl"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = pathlib.Path("/content/drive/MyDrive/EP3_SAPU"); destino.mkdir(parents=True, exist_ok=True)
    shutil.copy("ep3_resultados.zip", destino / "ep3_resultados.zip")
    print("✅ Copiado a", destino / "ep3_resultados.zip")
except Exception as e:
    print("Drive no disponible:", e)